# 🇮🇹 Italiano all'Università — Pimsleur-style audio lesson generator (B1)

**Lesson 1:** college life · learning · engineering · registering for classes · student–professor relationship

How the audio works:
* **English cue → pause → Italian answer → pause.** The first pause is for you to *say* the sentence, the second for you to *repeat* after the native speaker.
* **Pause lengths depend on the Italian.** Each pause is computed from the measured duration of the Italian clip:
  * answer pause = `PAUSE_SCALE × clamp(THINK_MS + SAY_FACTOR × d_it)`
  * repeat pause = `PAUSE_SCALE × clamp(REPEAT_BASE_MS + REPEAT_FACTOR × d_it)`
* **Two Italian voices** (female/male, Microsoft neural voices via `edge-tts`) plus an English narrator; falls back to `gTTS` automatically if a clip fails.
* Back-chaining (`ingegneria → gegneria → ...`), repeated words (`x2`, `x3`, optional slow repeat), spaced review of earlier sentences, role-play, and the dialogue at start and end.

**Run:** Runtime → Run all. The MP3 and a time-stamped transcript are downloaded at the end (~3–5 min synthesis for ~420 clips; the cache makes reruns fast).


In [ ]:
!pip -q install edge-tts gTTS pydub
# ffmpeg is pre-installed on Colab (pydub needs it)

## 1 · Settings

In [ ]:
#@title Voices, speed and pause settings { display-mode: "form" }
LESSON_NAME   = "Lesson01_Universita_B1"  #@param {type:"string"}
ENGINE        = "edge"   #@param ["edge", "gtts"]
VOICE_EN      = "en-US-AndrewNeural"  #@param ["en-US-AndrewNeural", "en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural"]
VOICE_IT_F    = "it-IT-ElsaNeural"    #@param ["it-IT-ElsaNeural", "it-IT-IsabellaNeural"]
VOICE_IT_M    = "it-IT-DiegoNeural"   #@param ["it-IT-DiegoNeural", "it-IT-GiuseppeNeural"]
EN_RATE       = "-5%"    #@param {type:"string"}
IT_RATE       = "-10%"   #@param {type:"string"}
SLOW_RATE     = "-35%"   #@param {type:"string"}

#@markdown **Pauses** (ms).  `d_it` = duration of the Italian clip.
PAUSE_SCALE    = 1.0    #@param {type:"slider", min:0.6, max:1.8, step:0.05}
THINK_MS       = 1500   #@param {type:"integer"}
SAY_FACTOR     = 1.5    #@param {type:"number"}
MIN_ANSWER_MS  = 2500   #@param {type:"integer"}
MAX_ANSWER_MS  = 12000  #@param {type:"integer"}
REPEAT_BASE_MS = 1000   #@param {type:"integer"}
REPEAT_FACTOR  = 1.3    #@param {type:"number"}
MIN_REPEAT_MS  = 1800   #@param {type:"integer"}
MAX_REPEAT_MS  = 10000  #@param {type:"integer"}
USE_PIANO      = True   #@param {type:"boolean"}

CFG = dict(ENGINE=ENGINE, VOICE_EN=VOICE_EN, VOICE_IT_F=VOICE_IT_F, VOICE_IT_M=VOICE_IT_M,
           EN_RATE=EN_RATE, IT_RATE=IT_RATE, SLOW_RATE=SLOW_RATE, PAUSE_SCALE=PAUSE_SCALE,
           THINK_MS=THINK_MS, SAY_FACTOR=SAY_FACTOR, MIN_ANSWER_MS=MIN_ANSWER_MS, MAX_ANSWER_MS=MAX_ANSWER_MS,
           REPEAT_BASE_MS=REPEAT_BASE_MS, REPEAT_FACTOR=REPEAT_FACTOR,
           MIN_REPEAT_MS=MIN_REPEAT_MS, MAX_REPEAT_MS=MAX_REPEAT_MS,
           NARRATION_GAP_MS=700, DIALOGUE_GAP_MS=600, SECTION_GAP_MS=1200,
           # dry-run speech-rate model (calibrated on the edge-tts voices above)
           IT_SYL_PER_S=4.6, EN_WORDS_PER_S=2.7, CLIP_PAD_MS=570,
           CACHE_DIR="/content/tts_cache", USE_PIANO=USE_PIANO,
           PIANO_URL="https://csee.umbc.edu/~simsek/shortpiano.mp3")

## 2 · TTS engine

In [ ]:
import os, re, json, time, hashlib, subprocess, tempfile, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pydub import AudioSegment

# ----------------------------------------------------------------------------
# Script tags (one per line; blank lines and lines starting with # are ignored)
#   (section) Title                 -> marker in the transcript only (no audio)
#   (narrator) English text         -> English narrator. {curly braces} = Italian
#                                      inserted inside the sentence, e.g.
#                                      "The word {ricevimento} means office hours."
#   (it F) testo  / (it M) testo    -> Italian line (dialogue), female / male voice
#   (repeat [x2] [slow] [F|M]) testo-> Italian model + pause for you to REPEAT.
#                                      x2/x3 = say it 2-3 times; slow = the extra
#                                      repetitions are spoken slowly.
#   (prompt [x2] [F|M]) English || Italiano
#                                   -> English cue, pause to ANSWER (length set by
#                                      the Italian), Italian answer, pause to REPEAT.
#   (build) zione | iscrizione      -> back-chaining: each piece + repeat pause
#   (pause 3)                       -> silence in seconds
#   (english) / (italian)           -> old tags still work (= narrator / it F)
# ----------------------------------------------------------------------------

VOWELS = "aeiouàèéìíòóùúy"

def italian_syllables(text):
    """Rough Italian syllable count (vowel groups); used for dry-run timing only."""
    words = re.findall(r"[A-Za-zÀ-ÿ']+", text.lower())
    n = 0
    for w in words:
        n += max(1, len(re.findall(f"[{VOWELS}]+", w)))
    return n


class PimsleurTTS:
    def __init__(self, cfg):
        self.cfg = cfg
        os.makedirs(cfg["CACHE_DIR"], exist_ok=True)
        self.piano_clip_path = self._download_piano_clip() if cfg.get("USE_PIANO") else None

    # ------------------------------------------------------------------ parsing
    def parse(self, script):
        events = []          # list of (kind, payload dict)
        voice_toggle = 0
        for raw in script.split("\n"):
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            m = re.match(r"^\(([^)]*)\)\s*(.*)$", line)
            if not m:
                print(f"⚠️ untagged line skipped: {line[:60]}")
                continue
            tag, body = m.group(1).strip().lower(), m.group(2).strip()
            words = tag.split()
            head = words[0]
            reps = 1
            for w in words:
                if re.fullmatch(r"x\d", w):
                    reps = int(w[1])
            slow = "slow" in words
            gender = "M" if "m" in words else ("F" if "f" in words else None)

            if head in ("section",):
                events.append(("section", {"title": body}))
            elif head in ("narrator", "english", "en"):
                events.append(("narrator", {"text": body}))
            elif head in ("it", "italian", "italiano"):
                events.append(("it", {"text": body, "g": gender or "F"}))
            elif head == "repeat":
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("repeat", {"text": body, "g": gender, "reps": reps, "slow": slow}))
            elif head == "prompt":
                if "||" not in body:
                    print(f"⚠️ prompt without '||' skipped: {body[:60]}"); continue
                en, it = [s.strip() for s in body.split("||", 1)]
                if gender is None:
                    gender = "FM"[voice_toggle % 2]; voice_toggle += 1
                events.append(("prompt", {"en": en, "it": it, "g": gender, "reps": reps, "slow": slow}))
            elif head == "build":
                pieces = [p.strip().lstrip("-") for p in body.split("|") if p.strip()]
                events.append(("build", {"pieces": pieces, "g": gender or "F"}))
            elif head == "pause":
                sec = float(words[1]) if len(words) > 1 else float(body or 1)
                events.append(("pause", {"ms": int(sec * 1000)}))
            else:
                print(f"⚠️ unknown tag ({tag}) skipped")
        return events

    # ------------------------------------------------------------------ pauses
    def answer_pause(self, it_ms):
        c = self.cfg
        p = c["THINK_MS"] + c["SAY_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_ANSWER_MS"]), c["MAX_ANSWER_MS"]))

    def repeat_pause(self, it_ms):
        c = self.cfg
        p = c["REPEAT_BASE_MS"] + c["REPEAT_FACTOR"] * it_ms
        return int(c["PAUSE_SCALE"] * min(max(p, c["MIN_REPEAT_MS"]), c["MAX_REPEAT_MS"]))

    # ------------------------------------------------------------------ TTS jobs
    def _voice(self, lang, g):
        c = self.cfg
        if lang == "en":
            return c["VOICE_EN"]
        return c["VOICE_IT_M"] if g == "M" else c["VOICE_IT_F"]

    @staticmethod
    def _split_narration(text):
        """'The word {casa} means house' -> [('en','The word'),('it','casa'),('en','means house')]"""
        parts = []
        for i, chunk in enumerate(re.split(r"\{(.*?)\}", text)):
            chunk = chunk.strip()
            if re.search(r"\w", chunk):          # skip punctuation-only pieces
                parts.append(("it" if i % 2 else "en", chunk))
        return parts

    def jobs_for(self, events):
        """All unique (text, lang, gender, slow) clips the lesson needs."""
        jobs = set()
        for kind, d in events:
            if kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    jobs.add((chunk, lang, "F", False))
            elif kind == "it":
                jobs.add((d["text"], "it", d["g"], False))
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                jobs.add((it, "it", d["g"], False))
                if d["slow"]:
                    jobs.add((it, "it", d["g"], True))
                if kind == "prompt":
                    jobs.add((d["en"], "en", "F", False))
            elif kind == "build":
                for p in d["pieces"]:
                    jobs.add((p, "it", d["g"], False))
        return sorted(jobs)

    def _cache_path(self, job):
        text, lang, g, slow = job
        key = json.dumps([text, self._voice(lang, g), slow, self.cfg["ENGINE"],
                          self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"],
                          self.cfg["SLOW_RATE"]])
        return os.path.join(self.cfg["CACHE_DIR"], hashlib.md5(key.encode()).hexdigest() + ".mp3")

    def _synth_edge(self, job, path):
        text, lang, g, slow = job
        rate = self.cfg["SLOW_RATE"] if slow else (self.cfg["EN_RATE"] if lang == "en" else self.cfg["IT_RATE"])
        cmd = ["edge-tts", f"--voice={self._voice(lang, g)}", f"--rate={rate}",
               f"--text={text}", f"--write-media={path}"]
        proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
        if proxy:                       # only matters behind a proxy; Colab has none
            cmd.insert(1, f"--proxy={proxy}")
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)

    def _synth_gtts(self, job, path):
        from gtts import gTTS
        text, lang, g, slow = job
        gTTS(text=text, lang=lang, slow=slow).save(path)

    @staticmethod
    def _clip_ok(job, path):
        """Reject empty / truncated clips (the online TTS sometimes returns silence)."""
        if not (os.path.exists(path) and os.path.getsize(path) > 0):
            return False
        try:
            a = AudioSegment.from_file(path)
        except Exception:
            return False
        n = italian_syllables(job[0]) if job[1] == "it" else 1.5 * len(job[0].split())
        return len(a) >= 1000 * n / 12 and a.dBFS > -60     # far faster than any voice = broken

    def _synth_one(self, job):
        path = self._cache_path(job)
        if self._clip_ok(job, path):
            return job, path, None
        err = None
        engines = [self.cfg["ENGINE"]] + (["gtts"] if self.cfg["ENGINE"] != "gtts" else [])
        for eng in engines:
            for attempt in range(4):
                try:
                    (self._synth_edge if eng == "edge" else self._synth_gtts)(job, path)
                    if self._clip_ok(job, path):
                        return job, path, None
                    err = f"{eng}: empty or truncated audio"
                except Exception as e:
                    err = f"{eng}: {e}"
                time.sleep(2 * (attempt + 1))      # back off: the free TTS service rate-limits bursts
        return job, None, err

    def synthesize_all(self, events, workers=6):
        jobs = self.jobs_for(events)
        print(f"🗣️  {len(jobs)} unique clips to synthesize (cached ones are reused)...")
        self.clips, self.failed = {}, []
        with ThreadPoolExecutor(workers) as ex:
            for i, (job, path, err) in enumerate(ex.map(self._synth_one, jobs), 1):
                if path:
                    self.clips[job] = AudioSegment.from_file(path)
                else:
                    self.failed.append((job, err))
                    self.clips[job] = AudioSegment.silent(duration=1000)
                if i % 100 == 0 or i == len(jobs):
                    print(f"   {i}/{len(jobs)} done")
        if self.failed:
            print(f"⚠️ {len(self.failed)} clips failed (replaced by 1 s silence), e.g.: {self.failed[0]}")

    def clip(self, text, lang, g="F", slow=False):
        return self.clips[(text, lang, g, slow)]

    # ------------------------------------------------------------------ assembly
    def assemble(self, events):
        c = self.cfg
        S = AudioSegment.silent
        out = S(duration=300)
        transcript = []

        def stamp(label, text):
            ms = len(out)
            transcript.append(f"[{ms//60000:02d}:{(ms//1000)%60:02d}] {label:9s} {text}")

        for kind, d in events:
            if kind == "section":
                stamp("SECTION", "=== " + d["title"] + " ===")
                out += S(duration=c["SECTION_GAP_MS"])
            elif kind == "pause":
                out += S(duration=int(d["ms"] * c["PAUSE_SCALE"]))
            elif kind == "narrator":
                stamp("NARRATOR", d["text"])
                for lang, chunk in self._split_narration(d["text"]):
                    out += self.clip(chunk, lang) + S(duration=120)
                out += S(duration=c["NARRATION_GAP_MS"])
            elif kind == "it":
                stamp(f"IT-{d['g']}", d["text"])
                out += self.clip(d["text"], "it", d["g"]) + S(duration=c["DIALOGUE_GAP_MS"])
            elif kind == "repeat":
                stamp("REPEAT", d["text"] + (f"  (x{d['reps']})" if d["reps"] > 1 else ""))
                normal = self.clip(d["text"], "it", d["g"])
                for r in range(d["reps"]):
                    seg = self.clip(d["text"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "prompt":
                stamp("PROMPT", d["en"] + "  ->  " + d["it"])
                normal = self.clip(d["it"], "it", d["g"])
                out += self.clip(d["en"], "en") + S(duration=self.answer_pause(len(normal)))
                for r in range(d["reps"]):
                    seg = self.clip(d["it"], "it", d["g"], True) if (d["slow"] and r > 0) else normal
                    out += seg + S(duration=self.repeat_pause(len(seg)))
            elif kind == "build":
                stamp("BUILD", " | ".join(d["pieces"]))
                for p in d["pieces"]:
                    seg = self.clip(p, "it", d["g"])
                    out += seg + S(duration=self.repeat_pause(len(seg)))
        return out, transcript

    # ------------------------------------------------------------------ dry run
    def estimate(self, events):
        """Duration estimate without any TTS (speech-rate model)."""
        c = self.cfg
        pad = c["CLIP_PAD_MS"]          # TTS clips carry some silence at both ends
        it_ms = lambda t, slow=False: pad + 1000 * italian_syllables(t) / (c["IT_SYL_PER_S"] * (0.75 if slow else 1))
        en_ms = lambda t: pad + 1000 * len(t.split()) / c["EN_WORDS_PER_S"]
        total, counts = 0.0, {}
        for kind, d in events:
            counts[kind] = counts.get(kind, 0) + 1
            if kind == "section":
                total += c["SECTION_GAP_MS"]
            elif kind == "pause":
                total += d["ms"] * c["PAUSE_SCALE"]
            elif kind == "narrator":
                for lang, chunk in self._split_narration(d["text"]):
                    total += (en_ms(chunk) if lang == "en" else it_ms(chunk)) + 120
                total += c["NARRATION_GAP_MS"]
            elif kind == "it":
                total += it_ms(d["text"]) + c["DIALOGUE_GAP_MS"]
            elif kind in ("repeat", "prompt"):
                it = d["text"] if kind == "repeat" else d["it"]
                if kind == "prompt":
                    total += en_ms(d["en"]) + self.answer_pause(it_ms(it))
                for r in range(d["reps"]):
                    m = it_ms(it, d["slow"] and r > 0)
                    total += m + self.repeat_pause(m)
            elif kind == "build":
                for p in d["pieces"]:
                    m = it_ms(p); total += m + self.repeat_pause(m)
        return total / 1000, counts

    # ------------------------------------------------------------------ main
    def _download_piano_clip(self):
        url = self.cfg.get("PIANO_URL")
        try:
            print("🎹 Downloading piano intro/outro clip...")
            p = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
            urllib.request.urlretrieve(url, p)
            return p
        except Exception as e:
            print(f"⚠️ Could not download piano clip: {e}")
            return None

    def build(self, script, output_file):
        events = self.parse(script)
        self.synthesize_all(events)
        print("🎧 Assembling lesson...")
        audio, transcript = self.assemble(events)
        if self.piano_clip_path and os.path.exists(self.piano_clip_path):
            piano = AudioSegment.from_file(self.piano_clip_path)[:10000]   # first 10 s
            intro = piano.fade_in(1500).fade_out(1500)
            outro = piano.fade_in(1000).fade_out(3000)
            audio = intro + AudioSegment.silent(duration=500) + audio + AudioSegment.silent(duration=800) + outro
        audio.export(output_file, format="mp3", bitrate="96k")
        txt = os.path.splitext(output_file)[0] + "_transcript.txt"
        with open(txt, "w", encoding="utf-8") as f:
            f.write("\n".join(transcript))
        print(f"✅ {output_file}: {len(audio)/60000:.1f} min   |   transcript: {txt}")
        return output_file, txt


## 3 · Lesson script

One line = one audio event. Tags:

| tag | what you hear |
|---|---|
| `(narrator) text {italiano}` | English narrator; text in `{ }` is read by the Italian voice |
| `(it F) …` / `(it M) …` | Italian line (dialogue), female / male |
| `(repeat [x2] [slow] [F/M]) …` | Italian model + repeat pause, 1–3 times; `slow` = extra repetitions spoken slowly |
| `(prompt [x2] [F/M]) English \|\| Italiano` | English cue → answer pause → Italian → repeat pause |
| `(build) zione \| iscrizione` | back-chaining, each piece + repeat pause |
| `(pause 2)` · `(section) Title` | silence · transcript marker |

Voices for `repeat`/`prompt` alternate F/M automatically unless `F` or `M` is given.

In [ ]:
LESSON = r"""
#####################################################################
# ITALIANO ALL'UNIVERSITÀ — Lesson 1 (B1)
# Theme: college life, learning, engineering, registration,
#        student–professor relationship
#####################################################################

(section) Opening
(narrator) Italian at the University. Lesson one.
(narrator) This lesson is for learners at level B1. You already know basic Italian. Today you will learn to talk about university life.
(narrator) You will hear a question in English. Then there is a pause. Answer out loud, in Italian. Then listen to the Italian speaker, and repeat after them.
(narrator) Speak out loud every time. Do not just think the answer.
(narrator) Listen to this conversation. Marco is an engineering student. He visits his professor, Professoressa Ferri, during her office hours.
(pause 1)
(it M) Buongiorno, professoressa. Grazie per il ricevimento.
(it F) Buongiorno, Marco. Si accomodi pure. Di che cosa ha bisogno?
(it M) Vorrei iscrivermi al suo corso di elettronica, ma la scadenza è già passata.
(it F) Ha parlato con la segreteria studenti?
(it M) Sì, ci sono andato ieri. Mi hanno detto che serve la sua firma.
(it F) Va bene. Ha già sostenuto l'esame di analisi?
(it M) Sì, l'ho superato a giugno, con ventotto.
(it F) Benissimo. Allora firmo il modulo. Però Le consiglio di frequentare anche il laboratorio.
(it M) Certo. Quando ci sono le lezioni di laboratorio?
(it F) Il martedì e il giovedì pomeriggio. Le serve altro?
(it M) No, grazie mille. È stata molto gentile.
(it F) Si figuri. Buono studio!
(pause 1.5)

(section) Part 1 — Office hours
(narrator) Let's look at this conversation, piece by piece.
(narrator) In Italy, a professor has fixed hours to meet students. This is called {il ricevimento}. It means office hours.
(narrator) Listen and repeat. Start from the end of the word.
(build) mento | vimento | ricevimento | il ricevimento
(repeat x2 slow) il ricevimento
(narrator) Marco thanks the professor for her time. He says: thank you for the office hour.
(repeat x2 slow) Grazie per il ricevimento.
(narrator) Marco speaks to a woman professor. He calls her {professoressa}. For a man, he would say {professore}.
(build) ressa | professoressa | Buongiorno, professoressa.
(prompt) How do you say: good morning, professor? Speak to a woman. || Buongiorno, professoressa.
(prompt) Now say: thank you for the office hour. || Grazie per il ricevimento.
(narrator) The professor uses the formal {Lei} with Marco. And Marco uses {Lei} with her. This is normal at an Italian university.
(narrator) She says: please, have a seat. Listen.
(repeat x2 slow) Si accomodi pure.
(narrator) Then she asks: what do you need?
(repeat x2 slow) Di che cosa ha bisogno?
(narrator) {avere bisogno di} means: to need.
(prompt) Ask your professor, formally: what do you need? || Di che cosa ha bisogno?
(prompt) Say: please, have a seat. Use the formal form. || Si accomodi pure.
(prompt) Say: I need your help. Use {il suo aiuto}. || Ho bisogno del suo aiuto.
(prompt) How do you say: thank you for the office hour? || Grazie per il ricevimento.

(narrator) Now, Marco's request. To say "I would like", use the conditional: {vorrei}.
(repeat x2) vorrei
(narrator) To enrol, or to register, is {iscriversi}. It is a reflexive verb.
(build) versi | iscriversi
(narrator) "I would like to register" is: {vorrei iscrivermi}. Notice the ending: {mi}.
(build) vermi | iscrivermi | vorrei iscrivermi
(prompt) Say: I would like to register. || Vorrei iscrivermi.
(prompt) Say: I would like to register for your course. Use the formal "your". || Vorrei iscrivermi al suo corso.
(prompt) Now: I would like to register for your electronics course. || Vorrei iscrivermi al suo corso di elettronica.
(narrator) The deadline is {la scadenza}.
(build) denza | scadenza | la scadenza
(prompt) Say: the deadline has already passed. || La scadenza è già passata.
(prompt) Say the full sentence: I would like to register for your course, but the deadline has already passed. || Vorrei iscrivermi al suo corso, ma la scadenza è già passata.
(prompt) What is "office hours"? || il ricevimento
(prompt) Ask the professor: what do you need? || Di che cosa ha bisogno?

(narrator) The student office is {la segreteria studenti}. Here you register, pay fees, and get documents.
(build) teria | segreteria | la segreteria studenti
(prompt) The professor asks: did you speak with the student office? || Ha parlato con la segreteria studenti?
(narrator) Marco answers: yes, I went there yesterday. The little word {ci} means "there".
(repeat x2 slow) Sì, ci sono andato ieri.
(narrator) A woman would say: {ci sono andata}.
(prompt) Say: I went there yesterday. || Ci sono andato ieri.
(narrator) Then: they told me that your signature is needed. {La firma} is the signature. {Serve} means: it is needed.
(repeat x2 slow) Mi hanno detto che serve la sua firma.
(prompt) Say: your signature is needed. Use the formal "your". || Serve la sua firma.
(prompt) Now say: they told me that your signature is needed. || Mi hanno detto che serve la sua firma.
(prompt) What is "the deadline"? || la scadenza
(prompt) Say: I would like to register for your course. || Vorrei iscrivermi al suo corso.

(narrator) Next, the professor asks about another exam. To take an exam is {sostenere un esame}. To pass an exam is {superare un esame}.
(repeat x2) sostenere un esame
(repeat x2) superare un esame
(prompt) Now you are the professor. Ask Marco, formally: have you already taken the analysis exam? || Ha già sostenuto l'esame di analisi?
(prompt) Answer: yes, I passed it in June. || Sì, l'ho superato a giugno.
(narrator) In Italy, exam grades go from eighteen to thirty. Thirty is the top grade. The best grade of all is {trenta e lode}: thirty with honours.
(repeat x2 slow) trenta e lode
(prompt) Say: I passed it with twenty-eight. || L'ho superato con ventotto.
(prompt) Say: I passed the exam with thirty with honours. || Ho superato l'esame con trenta e lode.
(prompt) Say: they told me that your signature is needed. || Mi hanno detto che serve la sua firma.

(narrator) The professor will sign the form. A form is {il modulo}. To sign is {firmare}.
(prompt) Say: then I'll sign the form. || Allora firmo il modulo.
(narrator) Then she gives advice. I advise you, formally, is {Le consiglio}. To attend is {frequentare}.
(build) tare | frequentare
(repeat x2 slow) Le consiglio di frequentare anche il laboratorio.
(prompt) Say: I advise you to attend the lab too. Use the formal form. || Le consiglio di frequentare anche il laboratorio.
(prompt) Ask: when are the lab classes? || Quando ci sono le lezioni di laboratorio?
(prompt) Answer: on Tuesday and Thursday afternoon. || Il martedì e il giovedì pomeriggio.
(prompt) Ask, formally: do you need anything else? || Le serve altro?
(prompt) Say: no, thank you very much. You have been very kind. Speak to a woman. || No, grazie mille. È stata molto gentile.
(narrator) She answers: {si figuri}. It means: don't mention it.
(repeat x2) Si figuri.
(narrator) And she says: {buono studio}. Enjoy your studying.
(prompt) Say: good morning, professor. Thank you for the office hour. || Buongiorno, professoressa. Grazie per il ricevimento.
(prompt) Say: I went there yesterday. || Ci sono andato ieri.
(prompt) Say: the deadline has already passed. || La scadenza è già passata.

(section) Part 2 — Registering for classes
(narrator) Now let's talk about registration. Registration, as a noun, is {l'iscrizione}.
(build) zione | crizione | iscrizione | l'iscrizione
(narrator) Each student has a study plan. It lists the courses for the year. It is called {il piano di studi}.
(repeat x2 slow) il piano di studi
(narrator) To fill in a form is {compilare}.
(prompt) Say: I have to fill in the study plan. || Devo compilare il piano di studi.
(prompt) Say: I have to fill in the study plan by Friday. Use {entro} for "by". || Devo compilare il piano di studi entro venerdì.
(narrator) Courses have credits. In Italian, {i crediti}. Students often say {CFU}.
(prompt) Ask: how many credits is this course worth? Use the verb {valere}. || Quanti crediti vale questo corso?
(prompt) Answer: this course is worth nine credits. || Questo corso vale nove crediti.
(narrator) A course can be required, {obbligatorio}, or elective, {a scelta}.
(build) torio | obbligatorio
(prompt) Ask: is it a required course or an elective? || È un corso obbligatorio o a scelta?
(prompt) Answer: it is required for engineering students. || È obbligatorio per gli studenti di ingegneria.
(prompt) Say: I would like to register for your course. || Vorrei iscrivermi al suo corso.
(prompt) What is "the student office"? || la segreteria studenti

(narrator) Before registration, you pay the university fees: {le tasse universitarie}.
(build) tarie | universitarie | le tasse universitarie
(prompt) Say: I have already paid the university fees. || Ho già pagato le tasse universitarie.
(narrator) Every student has a student number. In Italian, {il numero di matricola}. A first-year student is also called {una matricola}.
(repeat x2 slow) il numero di matricola
(prompt) The clerk asks you, formally: what is your student number? || Qual è il suo numero di matricola?
(prompt) Ask: where can I find the class schedule? Use {l'orario delle lezioni}. || Dove posso trovare l'orario delle lezioni?
(prompt) Answer: on the university website. || Sul sito dell'università.
(prompt) Say: the semester starts in October. || Il semestre comincia a ottobre.
(prompt) Say: the deadline has already passed. || La scadenza è già passata.
(prompt) Say: I advise you to attend the lab. Formal. || Le consiglio di frequentare il laboratorio.

(narrator) Now a short role-play. You are at the student office. Answer the clerk.
(it F) Buongiorno. Mi dica.
(prompt) Say: good morning. I would like some information about registration. || Buongiorno. Vorrei alcune informazioni sull'iscrizione.
(it F) Certo. Ha già pagato le tasse?
(prompt) Say: yes, I paid them last week. || Sì, le ho pagate la settimana scorsa.
(it F) Bene. Ha compilato il piano di studi?
(prompt) Say: not yet. When is the deadline? || Non ancora. Quando è la scadenza?
(it F) La scadenza è il trenta settembre.
(prompt) Say: perfect. Thank you very much. || Perfetto. Grazie mille.
(prompt) How do you say "the study plan"? || il piano di studi
(prompt) Ask: how many credits is this course worth? || Quanti crediti vale questo corso?

(section) Part 3 — Engineering and learning
(narrator) Now let's talk about your studies. Engineering is {ingegneria}.
(build) neria | gegneria | ingegneria
(repeat x3 slow) ingegneria
(prompt) Say: I study electronic engineering. || Studio ingegneria elettronica.
(prompt) Say: I am in my second year of engineering. || Sono al secondo anno di ingegneria.
(prompt) Ask a classmate, informally: what do you study? || Che cosa studi?
(prompt) Answer: I study mechanical engineering. || Studio ingegneria meccanica.
(prompt) Say: my favourite subject is physics. || La mia materia preferita è la fisica.
(prompt) Say: the course is required. || Il corso è obbligatorio.

(narrator) Now we talk about the past. For habits in the past, Italian uses the imperfect. For one finished event, it uses the passato prossimo.
(narrator) Listen: when I was in first year, I always studied in the library.
(repeat x2 slow) Quando ero al primo anno, studiavo sempre in biblioteca.
(prompt) Say: when I was in first year... || Quando ero al primo anno...
(prompt) Say: I always studied in the library. || Studiavo sempre in biblioteca.
(prompt) Now the whole sentence: when I was in first year, I always studied in the library. || Quando ero al primo anno, studiavo sempre in biblioteca.
(narrator) Now one event. Yesterday I studied until midnight.
(prompt) Say: yesterday I studied until midnight. || Ieri ho studiato fino a mezzanotte.
(prompt) Say: I passed the exam with twenty-eight. || Ho superato l'esame con ventotto.
(prompt) Say: I have already paid the university fees. || Ho già pagato le tasse universitarie.

(narrator) Engineering students do many projects. A group project is {un progetto di gruppo}.
(repeat x2 slow) un progetto di gruppo
(prompt) Say: we work in a group on a circuit. || Lavoriamo in gruppo su un circuito.
(prompt) Say: last semester we designed a circuit. Use {progettare}. || Il semestre scorso abbiamo progettato un circuito.
(prompt) Say: the experiment did not work. || L'esperimento non ha funzionato.
(prompt) Say: we had to repeat the measurements. || Abbiamo dovuto ripetere le misure.
(narrator) Many Italian exams have a written part, {lo scritto}, and an oral part, {l'orale}.
(prompt) Say: I passed the written exam. || Ho passato lo scritto.
(prompt) Say: I passed the written exam, but I still have to do the oral. || Ho passato lo scritto, ma devo ancora fare l'orale.
(prompt) Ask your professor, formally: when is the next exam session? Use {l'appello}. || Quando è il prossimo appello?
(prompt) Say: I have to fill in the study plan by Friday. || Devo compilare il piano di studi entro venerdì.

(narrator) Now the future. A degree is {la laurea}. To graduate is {laurearsi}.
(build) rearsi | laurearsi
(prompt) Say: I will graduate next year. || Mi laureerò l'anno prossimo.
(narrator) In Italy, the bachelor's degree is {la laurea triennale}, three years. The master's is {la laurea magistrale}.
(prompt) Say: after the bachelor's degree, I will do the master's. || Dopo la laurea triennale, farò la magistrale.
(narrator) The thesis is {la tesi}. The thesis advisor is {il relatore}, or {la relatrice} for a woman.
(prompt) Say: I would like to do my thesis on photonics. || Vorrei fare la tesi sulla fotonica.
(prompt) Ask, formally: could you be my thesis advisor? Speak to a woman. || Potrebbe essere la mia relatrice?
(prompt) Say: when I was in first year, I always studied in the library. || Quando ero al primo anno, studiavo sempre in biblioteca.
(prompt) What is "office hours"? || il ricevimento

(section) Part 4 — Student and professor
(narrator) Now let's talk to professors politely. Use the conditional of {potere}: {potrebbe}. It means: could you.
(repeat x2) potrebbe
(prompt) Ask, formally: could you explain this exercise again? || Potrebbe spiegarmi di nuovo questo esercizio?
(build) cizio | esercizio | questo esercizio
(prompt) Say: I didn't understand the proof. Use {la dimostrazione}. || Non ho capito la dimostrazione.
(prompt) Ask, formally: may I ask you a question? || Posso farle una domanda?
(prompt) Say: excuse me for being late. Formal. || Mi scusi per il ritardo.
(prompt) Say: I will write you an email. Formal. || Le scrivo un'email.
(prompt) Say: I need a recommendation letter. || Ho bisogno di una lettera di raccomandazione.
(prompt) Ask, formally: could you write me a recommendation letter? || Potrebbe scrivermi una lettera di raccomandazione?
(prompt) Say: I passed the written exam, but I still have to do the oral. || Ho passato lo scritto, ma devo ancora fare l'orale.

(narrator) Now give your opinion. After {penso che}, I think that, Italian uses the subjunctive.
(narrator) Listen: I think that the exam is difficult.
(repeat x2 slow) Penso che l'esame sia difficile.
(prompt) Say: I think that the exam is difficult. || Penso che l'esame sia difficile.
(prompt) Say: I think the professor is right. || Penso che il professore abbia ragione.
(prompt) Say: I think the course is very useful. || Penso che il corso sia molto utile.
(narrator) After {è importante che}, it is important that, you also use the subjunctive.
(prompt) Say: it is important that students ask questions. || È importante che gli studenti facciano domande.
(prompt) Say: it is important that you attend the lab. Speak informally to a friend. || È importante che tu frequenti il laboratorio.
(prompt) Say: our professor is very available. || La nostra professoressa è molto disponibile.
(prompt) Say: and she is very well prepared. || Ed è molto preparata.
(narrator) To use {Lei} with someone is {dare del Lei}. To use {tu} is {dare del tu}.
(prompt) Say: in Italy, students use "Lei" with professors. || In Italia, gli studenti danno del Lei ai professori.
(prompt) Say: but with classmates, we use "tu". || Ma con i compagni di corso, diamo del tu.
(prompt) Say: I would like to do my thesis on photonics. || Vorrei fare la tesi sulla fotonica.
(prompt) Ask: is it a required course or an elective? || È un corso obbligatorio o a scelta?

(section) Part 5 — Student life
(narrator) Finally, daily student life. The university cafeteria is {la mensa}. A classmate is {un compagno di corso}.
(prompt) Say: I have lunch at the cafeteria with my classmates. || Pranzo in mensa con i miei compagni di corso.
(narrator) A roommate, or flatmate, is {un coinquilino}.
(build) lino | quilino | coinquilino
(prompt) Say: I live with two flatmates near the university. || Abito con due coinquilini vicino all'università.
(prompt) Say: last year I did Erasmus in Spain. || L'anno scorso ho fatto l'Erasmus in Spagna.
(prompt) Say: I won a scholarship. Use {una borsa di studio}. || Ho vinto una borsa di studio.
(prompt) Say: the library is always full during the exam period. Use {la sessione}. || La biblioteca è sempre piena durante la sessione.
(prompt) Say: I think the exam is difficult. || Penso che l'esame sia difficile.
(prompt) Ask, formally: could you explain this exercise again? || Potrebbe spiegarmi di nuovo questo esercizio?
(prompt) Say: I will graduate next year. || Mi laureerò l'anno prossimo.

(section) Part 6 — Review
(narrator) Let's review. Answer quickly. Then repeat.
(prompt) Say: I would like to register for your course, but the deadline has already passed. || Vorrei iscrivermi al suo corso, ma la scadenza è già passata.
(prompt) Say: they told me that your signature is needed. || Mi hanno detto che serve la sua firma.
(prompt) Say: I passed it with twenty-eight. || L'ho superato con ventotto.
(prompt) Say: I advise you to attend the lab too. Formal. || Le consiglio di frequentare anche il laboratorio.
(prompt) Say: I have to fill in the study plan by Friday. || Devo compilare il piano di studi entro venerdì.
(prompt) Ask: how many credits is this course worth? || Quanti crediti vale questo corso?
(prompt) Say: I am in my second year of engineering. || Sono al secondo anno di ingegneria.
(prompt) Say: last semester we designed a circuit. || Il semestre scorso abbiamo progettato un circuito.
(prompt) Ask, formally: may I ask you a question? || Posso farle una domanda?
(prompt) Say: it is important that students ask questions. || È importante che gli studenti facciano domande.
(prompt) Say: I live with two flatmates near the university. || Abito con due coinquilini vicino all'università.
(prompt) Say: you have been very kind. Speak to a woman. || È stata molto gentile.

(narrator) Now, you are Marco. Answer the professor. Listen.
(it F) Buongiorno. Si accomodi pure. Di che cosa ha bisogno?
(prompt M) Say: I would like to register for your course, but the deadline has already passed. || Vorrei iscrivermi al suo corso, ma la scadenza è già passata.
(it F) Ha parlato con la segreteria studenti?
(prompt M) Say: yes, I went there yesterday. They told me that your signature is needed. || Sì, ci sono andato ieri. Mi hanno detto che serve la sua firma.
(it F) Va bene. Ha già sostenuto l'esame di analisi?
(prompt M) Say: yes, I passed it in June, with thirty with honours. || Sì, l'ho superato a giugno, con trenta e lode.
(it F) Complimenti! Allora firmo il modulo.
(prompt M) Say: thank you very much. You have been very kind. || Grazie mille. È stata molto gentile.

(narrator) Now listen to the first conversation again. You should understand all of it.
(pause 1)
(it M) Buongiorno, professoressa. Grazie per il ricevimento.
(it F) Buongiorno, Marco. Si accomodi pure. Di che cosa ha bisogno?
(it M) Vorrei iscrivermi al suo corso di elettronica, ma la scadenza è già passata.
(it F) Ha parlato con la segreteria studenti?
(it M) Sì, ci sono andato ieri. Mi hanno detto che serve la sua firma.
(it F) Va bene. Ha già sostenuto l'esame di analisi?
(it M) Sì, l'ho superato a giugno, con ventotto.
(it F) Benissimo. Allora firmo il modulo. Però Le consiglio di frequentare anche il laboratorio.
(it M) Certo. Quando ci sono le lezioni di laboratorio?
(it F) Il martedì e il giovedì pomeriggio. Le serve altro?
(it M) No, grazie mille. È stata molto gentile.
(it F) Si figuri. Buono studio!
(pause 1.5)
(narrator) This is the end of lesson one. Listen to it again tomorrow, before you start lesson two. {Buono studio!}
"""

## 4 · Dry run (no audio) — check length before synthesizing

In [ ]:
tts = PimsleurTTS(CFG)
events = tts.parse(LESSON)
sec, counts = tts.estimate(events)
print(f"Estimated length: {sec/60:.1f} min   (PAUSE_SCALE={PAUSE_SCALE})")
print(counts)
print(f"Unique clips to synthesize: {len(tts.jobs_for(events))}")

## 5 · Build the MP3 + transcript

In [ ]:
mp3, txt = tts.build(LESSON, f"{LESSON_NAME}.mp3")
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3); files.download(txt)
except ImportError:
    pass

## 6 · Making the next lesson

Copy this notebook, change `LESSON_NAME`, and replace the `LESSON` script. A prompt that works well with an LLM:

> Write a ~45-minute Pimsleur-style Italian lesson at level **B1** on *[topic]*, in this tag format: `(narrator)`, `(it F)`, `(it M)`, `(repeat x2 slow)`, `(prompt) English || Italiano`, `(build) a | b | c`, `(pause n)`, `(section)`. Narration in plain short English sentences; put Italian words inside narration in `{braces}`. Start with a 10–14-line dialogue, break it down with back-chaining for long words, ~120 prompts, re-prompt each new sentence 2–4 times at growing intervals (spaced recall), recycle 10–15 sentences from Lesson *N–1*, include one role-play, and replay the dialogue at the end.

Tips
* Too fast/slow for you? Move `PAUSE_SCALE` (it scales every learner pause) — no need to edit the script.
* Long sentences cap at `MAX_ANSWER_MS`; raise it if you need more time on long prompts.
* `edge-tts` is an unofficial client of the Microsoft Edge read-aloud service; if it breaks, set `ENGINE="gtts"` (single Italian voice, no male/female distinction).
